Desenvolver um script que consulte duma API publica (Correios) e retorne as informações de endereço formatadas em JSON de maneira legivel. Use tratamento de execerção

In [5]:
import requests
import json

def consultar_cep_e_formatar_json(cep):
    """
    Consulta o CEP na API ViaCEP e retorna as informações de endereço
    formatadas em JSON de maneira legível, com tratamento de exceções.
    """
    url = f"https://viacep.com.br/ws/{cep}/json/" # URL correta da ViaCEP

    print(f"Consultando o CEP {cep} na URL: {url}\n")

    try:
        response = requests.get(url, timeout=5) # Adiciona um timeout para a requisição
        response.raise_for_status() # Lança exceções para status HTTP de erro (4xx, 5xx)

        data = response.json()

        # Verifica se a API ViaCEP retornou um erro (ex: CEP não encontrado)
        if 'erro' in data and data['erro'] == True:
            print(f"Erro: CEP '{cep}' não encontrado ou inválido.")
            return json.dumps({"status": "erro", "mensagem": f"CEP '{cep}' não encontrado ou inválido."}, indent=2, ensure_ascii=False)

        # Formata as informações em um dicionário para JSON
        endereco = {
            "cep": data.get("cep"),
            "logradouro": data.get("logradouro"),
            "complemento": data.get("complemento"),
            "bairro": data.get("bairro"),
            "localidade": data.get("localidade"),
            "uf": data.get("uf"),
            "ibge": data.get("ibge"),
            "gia": data.get("gia"),
            "ddd": data.get("ddd"),
            "siafi": data.get("siafi")
        }

        # Retorna o JSON formatado e legível
        return json.dumps(endereco, indent=2, ensure_ascii=False)

    except requests.exceptions.Timeout:
        print("Erro: A requisição excedeu o tempo limite. Tente novamente mais tarde.")
        return json.dumps({"status": "erro", "mensagem": "Requisição excedeu o tempo limite."}, indent=2, ensure_ascii=False)
    except requests.exceptions.ConnectionError:
        print("Erro: Não foi possível conectar ao servidor da ViaCEP. Verifique sua conexão com a internet.")
        return json.dumps({"status": "erro", "mensagem": "Falha na conexão com a internet ou servidor ViaCEP."}, indent=2, ensure_ascii=False)
    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP: {http_err} - Status Code: {response.status_code}")
        return json.dumps({"status": "erro", "mensagem": f"Erro HTTP {response.status_code}: {http_err}"}, indent=2, ensure_ascii=False)
    except json.JSONDecodeError:
        print("Erro: Não foi possível decodificar a resposta JSON da API. A resposta pode estar malformada.")
        print(f"Resposta bruta: {response.text}")
        return json.dumps({"status": "erro", "mensagem": "Resposta JSON inválida da API."}, indent=2, ensure_ascii=False)
    except requests.exceptions.RequestException as err:
        print(f"Ocorreu um erro inesperado na requisição: {err}")
        return json.dumps({"status": "erro", "mensagem": f"Erro inesperado na requisição: {err}"}, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")
        return json.dumps({"status": "erro", "mensagem": f"Erro inesperado: {e}"}, indent=2, ensure_ascii=False)

# --- Exemplo de uso com input do usuário ---
input_cep = input("Por favor, digite o CEP que deseja consultar (apenas números): ")

# Limpa o CEP removendo caracteres não numéricos
input_cep_limpo = ''.join(filter(str.isdigit, input_cep))

if len(input_cep_limpo) == 8:
    print(f"\n--- Consultando CEP: {input_cep_limpo} ---")
    resultado_json = consultar_cep_e_formatar_json(input_cep_limpo)
    if resultado_json:
        print("\nResultado JSON:")
        print(resultado_json)
        print("------------------------------------------\n")
else:
    print("Erro: O CEP deve conter 8 dígitos numéricos. Por favor, tente novamente.")


Por favor, digite o CEP que deseja consultar (apenas números): 73350102

--- Consultando CEP: 73350102 ---
Consultando o CEP 73350102 na URL: https://viacep.com.br/ws/73350102/json/


Resultado JSON:
{
  "cep": "73350-102",
  "logradouro": "Quadra 1 Conjunto B",
  "complemento": "",
  "bairro": "Setor Residencial Leste (Planaltina)",
  "localidade": "Brasília",
  "uf": "DF",
  "ibge": "5300108",
  "gia": "",
  "ddd": "61",
  "siafi": "9701"
}
------------------------------------------

